In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import math
import numpy as np
import polars as pl
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cuda


In [3]:
train = pl.scan_parquet(PROCESSED_PATH / "train.parquet")
validation_ground_truth = pl.read_parquet(PROCESSED_PATH / "validation_ground_truth.parquet")
customer_mapping = pl.read_parquet(PROCESSED_PATH / "customer_mapping.parquet")
article_mapping = pl.read_parquet(PROCESSED_PATH / "article_mapping.parquet")

NUM_USERS = customer_mapping.height + 1
NUM_ITEMS = article_mapping.height + 1

print("Users:", NUM_USERS - 1)
print("Items:", NUM_ITEMS - 1)

Users: 1371980
Items: 105542


In [4]:
train_pairs = (
    train
    .select("customer_idx", "article_idx")
    .unique()
    .collect()
)

print("Unique user-item pairs:", train_pairs.height)

Unique user-item pairs: 26882193


In [5]:
users = torch.from_numpy(train_pairs["customer_idx"].to_numpy().astype(np.int64))
items = torch.from_numpy(train_pairs["article_idx"].to_numpy().astype(np.int64))

dataset = TensorDataset(users, items)

print("Training pairs:", len(dataset))

Training pairs: 26882193


In [6]:
del train_pairs

## BPR

In [7]:
class BPR(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()

        self.user_embedding = nn.Embedding(num_users, embedding_dim, sparse=True)
        self.item_embedding = nn.Embedding(num_items, embedding_dim, sparse=True)

        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)

    def forward(self, users, positive, negative):
        user = self.user_embedding(users)
        pos = self.item_embedding(positive)
        neg = self.item_embedding(negative)

        positive_score = (user * pos).sum(dim=1)
        negative_score = (user * neg).sum(dim=1)

        return user, pos, neg, positive_score, negative_score

In [8]:
EMBEDDING_DIM = 64
BATCH_SIZE = 16384
LEARNING_RATE = 0.001
REG = 1e-6
EPOCHS = 3

model = BPR(NUM_USERS, NUM_ITEMS, EMBEDDING_DIM).to(DEVICE)
optimizer = torch.optim.SparseAdam(model.parameters(), lr=LEARNING_RATE)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [9]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch_users, positive_items in loader:
        batch_users = batch_users.to(DEVICE, non_blocking=True)
        positive_items = positive_items.to(DEVICE, non_blocking=True)

        negative_items = torch.randint(1, NUM_ITEMS, positive_items.shape, device=DEVICE)
        negative_items = torch.where(negative_items == positive_items, negative_items % (NUM_ITEMS - 1) + 1, negative_items)

        user, pos, neg, positive_score, negative_score = model(
            batch_users,
            positive_items,
            negative_items
        )

        bpr_loss = -torch.nn.functional.logsigmoid(
            positive_score - negative_score
        ).mean()

        reg_loss = REG * (
            user.pow(2).sum(dim=1)
            + pos.pow(2).sum(dim=1)
            + neg.pow(2).sum(dim=1)
        ).mean()

        loss = bpr_loss + reg_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_users)

    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {total_loss / len(dataset):.4f}")

Epoch 1/3 - loss: 0.5711
Epoch 2/3 - loss: 0.3157
Epoch 3/3 - loss: 0.2750


In [10]:
CHECKPOINTS_PATH = PROJECT_PATH / "checkpoints"
CHECKPOINTS_PATH.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "embedding_dim": EMBEDDING_DIM,
    "num_users": NUM_USERS,
    "num_items": NUM_ITEMS
}, CHECKPOINTS_PATH / "bpr.pt")

In [11]:
from datetime import date, timedelta

In [12]:
VAL_START = date(2020, 9, 9)
K = 12

recent_top12 = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=14))
    .group_by("article_idx")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(K)
    .collect()["article_idx"]
    .to_list()
)

recent_top12

[103794,
 67523,
 67544,
 101719,
 53893,
 103797,
 104046,
 94675,
 105147,
 103795,
 103796,
 101368]

In [13]:
train_user_ids = train.select("customer_idx").unique().collect()["customer_idx"].to_numpy()
candidate_items = train.select("article_idx").unique().collect()["article_idx"].to_numpy()

has_history = np.zeros(NUM_USERS, dtype=bool)
has_history[train_user_ids] = True

print("Train users:", len(train_user_ids))
print("Candidate items:", len(candidate_items))

Train users: 1351314
Candidate items: 102967


In [14]:
val_user_ids = validation_ground_truth["customer_idx"].to_numpy()
candidate_items_tensor = torch.from_numpy(candidate_items.astype(np.int64)).to(DEVICE)

model.eval()

with torch.no_grad():
    item_embeddings = model.item_embedding(candidate_items_tensor)

In [15]:
EVAL_BATCH_SIZE = 512
predictions = []

with torch.no_grad():
    for start in range(0, len(val_user_ids), EVAL_BATCH_SIZE):
        batch_ids = val_user_ids[start:start + EVAL_BATCH_SIZE]
        batch_users = torch.from_numpy(batch_ids.astype(np.int64)).to(DEVICE)

        user_embeddings = model.user_embedding(batch_users)
        scores = user_embeddings @ item_embeddings.T

        top_indices = scores.topk(K, dim=1).indices
        batch_predictions = candidate_items_tensor[top_indices].cpu().tolist()

        for i, user_id in enumerate(batch_ids):
            if not has_history[user_id]:
                batch_predictions[i] = recent_top12

        predictions.extend(batch_predictions)

print("Predictions:", len(predictions))
print(predictions[:3])

Predictions: 72019
[[53893, 1714, 53894, 70222, 3712, 1715, 24838, 2237, 58492, 1470, 53895, 24837], [53893, 1714, 70222, 3712, 1715, 53894, 1470, 24838, 58492, 47296, 2237, 20851], [53893, 53894, 1714, 70222, 24838, 1715, 3712, 2237, 53895, 2253, 14254, 16004]]


In [16]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)
    if not actual:
        return 0.0

    score = 0.0
    hits = 0
    seen = set()

    for i, item in enumerate(predicted[:k], 1):
        if item in actual and item not in seen:
            hits += 1
            score += hits / i
        seen.add(item)

    return score / min(len(actual), k)


def recall_at_k(actual, predicted, k=12):
    actual = set(actual)
    if not actual:
        return 0.0

    return len(actual.intersection(predicted[:k])) / len(actual)


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)
    if not actual:
        return 0.0

    dcg = sum(1 / math.log2(i + 2) for i, item in enumerate(predicted[:k]) if item in actual)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(actual), k)))

    return dcg / idcg

In [17]:
def evaluate(actuals, predictions, catalog_size, k=12):
    return {
        "MAP@12": sum(average_precision_at_k(actual, predicted, k) for actual, predicted in zip(actuals, predictions)) / len(actuals),
        "Recall@12": sum(recall_at_k(actual, predicted, k) for actual, predicted in zip(actuals, predictions)) / len(actuals),
        "NDCG@12": sum(ndcg_at_k(actual, predicted, k) for actual, predicted in zip(actuals, predictions)) / len(actuals),
        "Coverage": len({item for predicted in predictions for item in predicted[:k]}) / (NUM_ITEMS - 1)
    }

In [18]:
actuals = validation_ground_truth["actual"].to_list()

bpr_metrics = evaluate(actuals, predictions, NUM_ITEMS - 1)

bpr_metrics

{'MAP@12': 0.0037134082292341244,
 'Recall@12': 0.010272175387045808,
 'NDCG@12': 0.006925831334384407,
 'Coverage': 0.010668738511682553}

In [19]:
comparison = pl.DataFrame([
    {
        "model": "Personal history 56d + recent popularity 14d",
        "MAP@12": 0.025170,
        "Recall@12": 0.049558,
        "NDCG@12": 0.036092,
        "Coverage": 0.218965
    },
    {
        "model": "BPR",
        **bpr_metrics
    }
]).sort("MAP@12", descending=True)

comparison

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""Personal history 56d + recent …",0.02517,0.049558,0.036092,0.218965
"""BPR""",0.003713,0.010272,0.006926,0.010669
